# MBRL Curvature — Colab GPU launcher

Runs the GPU side of the project (Mode A: full loop; Mode B: train on shards
collected locally). Designed for Colab Pro session death: checkpoints push to
W&B every `checkpoint.every` updates and `checkpoint.resume=auto` picks up the
newest one on relaunch — just re-run all cells.

**Runtime → Change runtime type → A100** (L4/T4 fine for Pendulum-class runs).

In [2]:
# 1. GPU sanity
import torch
print(torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

2.11.0+cpu | cuda: False | -


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
# 2. Get the code — clones fresh every run (clears stale dirs), surfaces git
# errors, then cds to wherever pyproject.toml lives. Private repo? Use
# REPO_URL = "https://<TOKEN>@github.com/you/repo.git" (fine-grained PAT, read-only).
import os, shutil, subprocess
REPO_URL = "https://github.com/BABYZ3US/MBRL.git"   # <-- set me

os.chdir("/content")
if REPO_URL:
    shutil.rmtree("/content/_repo", ignore_errors=True)
    r = subprocess.run(["git", "clone", REPO_URL, "/content/_repo"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{r.stderr}")

hits = subprocess.run(
    ["find", "/content", "-maxdepth", "4", "-name", "pyproject.toml",
     "-not", "-path", "*/.*"], capture_output=True, text=True).stdout.split()
assert hits, "no pyproject.toml under /content — set REPO_URL above"
PROJECT = os.path.dirname(sorted(hits, key=len)[0])
os.chdir(PROJECT)
print("project:", PROJECT)


In [ ]:
!pip -q install -e ".[mujoco]" 2>&1 | tail -1
!python -c "import mbrl, gymnasium; print('mbrl + gymnasium import ok')"


In [4]:
# 3. W&B login — works on any frontend.
import os
key = os.environ.get("WANDB_API_KEY")
if not key:
    try:                       # web Colab Secrets (key icon), if available
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
    except Exception:
        from getpass import getpass
        key = getpass("WANDB_API_KEY (from wandb.ai/authorize): ")
os.environ["WANDB_API_KEY"] = key
import wandb; wandb.login()

In [5]:
# 4. Launch — paste any preset/command here. After a runtime recycle just
# re-run the preamble (cells 1-3) and re-run this; checkpoint.resume=auto
# continues each run from its W&B artifact lineage.
#
# Phase 0 regression:   --preset colab_recipe  --overrides env=halfcheetah model.latent_dim=17
# Phase 1 breadth:      --preset colab_recipe  --overrides env=walker2d model.latent_dim=17
#                       --preset colab_control --overrides env=walker2d --seeds 0
# Estimator h2h:        --preset colab_estimator --overrides env=halfcheetah
# Schedule final:       --preset colab_schedule_final --overrides env=halfcheetah
# Shiny + auto-dose:    --overrides experiment.name=shiny-hc env=halfcheetah \
#                         penalty.auto_dose.enabled=true penalty.auto_dose.warmup_updates=2000
!python scripts/parallel_runs.py --preset colab_recipe \
    --overrides env=halfcheetah model.latent_dim=17 --seeds 0 1 2 --jobs 3

In [ ]:
# 4b. Mode B — import locally-collected replay shards first (optional)
# import wandb
# art = wandb.Api().artifact("you/mbrl-curvature/replay-HalfCheetah-v5:latest")
# shard_dir = art.download()
# then pass +buffer.shards=$shard_dir to train.py (wire-up in train.py when needed)

In [ ]:
# 5. Join a W&B sweep (GPU agent). Local CPU agents can join the same sweep id.
# SWEEP_ID = "you/mbrl-curvature/abc123"
# !wandb agent $SWEEP_ID

In [ ]:
# 6. Keep-alive / disconnect drill: simulate a kill and verify resume works.
# !timeout 120 python scripts/train.py seed=0   # dies after 2 min
# !python scripts/train.py seed=0 checkpoint.resume=auto   # must continue, not restart